In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# GPUs, Devices, and Memory

So far every tensor in this book has lived in main memory and every computation
has run on the CPU. Training at any interesting scale runs on an accelerator,
almost always a CUDA GPU, and that changes two things at once. First, every
tensor and every parameter now has a *home*, a device, and operations only
combine tensors that live on the same one; placing data well is your job.
Second, the GPU's memory is small compared with main memory, typically tens of
gigabytes, and it is the resource you will exhaust first: the daily question of
a single-GPU builder is not "is my model correct?" but "does it fit?". This
section covers both skills: naming devices and placing tensors and models on
them, then measuring what actually fills GPU memory during training, trading
compute for memory when it does not fit, and keeping the device busy once it
does. None of the code requires a GPU to run; the helpers we define fall back
to the CPU, and the memory measurements simply report more interesting numbers
when a GPU is present.

In [ ]:
import time
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Devices

In JAX every array lives on a device, and `jax.devices(backend)` returns the
device handles of a backend. `jax.devices('cpu')[0]` is the CPU, and it stands
for *all* physical CPU cores and all of main memory; `jax.devices('gpu')[i]`
is one specific CUDA card and its own memory. To see what your machine has,
run `nvidia-smi` in a shell: it lists every card, its memory, and what is
currently running on it. We wrap device lookup in two helpers that the rest
of the book uses.

In [ ]:
def cpu():
    """Get the CPU device."""
    return jax.devices('cpu')[0]

def gpu(i=0):
    """Get a GPU device."""
    return jax.devices('gpu')[i]

cpu()

Note that a JAX device object is a handle to hardware that actually exists:
`jax.devices('gpu')` raises a `RuntimeError` when no GPU backend is present,
rather than returning a placeholder (which is why the demo above calls only
`cpu()`). Our next helper therefore counts GPUs through a `try`/`except`.

We can query how many GPUs are actually available.

In [ ]:
def num_gpus():
    """Get the number of available GPUs."""
    try:
        return jax.device_count('gpu')
    except:
        return 0  # No GPU backend found

num_gpus()

Now we define two convenient functions that allow us
to run code even if the requested GPUs do not exist.

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    devices = [gpu(i) for i in range(num_gpus())]
    return devices if devices else [cpu()]

try_gpu(), try_gpu(10), try_all_gpus()

CUDA is not the only backend JAX understands: the same program runs on CPU,
GPU, or TPU, and plain `jax.devices()` lists the devices of the *default*
backend, which is the best accelerator present. We keep our own helpers
anyway: they give all four of the book's implementations one vocabulary, and
they degrade to the CPU instead of failing, which is exactly what lets this
book's code run unchanged on a laptop. The book standardizes on CPU plus
CUDA; everything below transfers to TPUs with no change beyond the backend
name.

## Tensors, Models, and Devices

By default, arrays are created on the *default device*, the first accelerator
when one exists and the CPU otherwise. We can query where an array lives.

In [ ]:
x = jnp.array([1, 2, 3])
x.device

To place an array somewhere specific, `jax.device_put(x, device)` copies it
there, and the result is *committed* to that device: JAX will keep it and
every computation on it exactly where you said. Arrays created without an
explicit placement, like `x` above, are *uncommitted*; they sit on the default
device but JAX may treat them as movable when they meet committed data.

In [ ]:
X = jax.device_put(jnp.ones((2, 3)), try_gpu())
X

On a machine with two GPUs we can put a second tensor on the second card
(on your machine, `try_gpu(1)` substitutes whatever is available).

In [ ]:
Y = jax.device_put(jax.random.uniform(jax.random.key(0), (2, 3)),
                   try_gpu(1))
Y

### Copying Between Devices

Whenever we operate on multiple arrays, they need to be on the same device.
Uncommitted arrays are flexible: JAX decides where the result lives. But `X`
is committed to the first GPU and `Y` to the second, and adding them raises an
error about arrays on different devices, probably the most common error
message of a beginner's GPU life. JAX refuses to guess between two explicit
placements: an implicit copy would hide a slow bus transfer inside an
innocent-looking `+`, and you would never find it. Instead we copy
explicitly, as in the figure, and then add.

![X lives on GPU 0 and Y on GPU 1; X.to(gpu(1)) makes a copy Z on GPU 1 (dashed), and Y + Z then runs entirely on GPU 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-copyto.svg)

In [ ]:
Z = jax.device_put(X, try_gpu(1))
print(X)
print(Z)

Now that `Z` and `Y` live on the same device, we can add them.

In [ ]:
Y + Z

What if `Z` already lives on the target device? Then `device_put` skips the
transfer: the array it returns shares `Z`'s buffer, so calling it defensively
costs nothing. Comparing the buffer addresses confirms that no copy was made.

In [ ]:
Z2 = jax.device_put(Z, try_gpu(1))
Z2.unsafe_buffer_pointer() == Z.unsafe_buffer_pointer()

The reason for keeping every copy explicit is the cost model. A
modern GPU multiplies matrices hundreds of times faster than it can receive
data over the PCIe bus, so a transfer in the wrong place can erase the
speedup you bought the GPU for. The discipline that follows is simple: move data to the device
once, at the boundary of the computation, and keep everything inside the
training loop on one device. Many small transfers are worse than one big one,
and a transfer hidden in an inner loop is worst of all.

### Models on a Device

An NNX model owns a state tree, and `jax.device_put` accepts that tree. Update
the model with the placed state and every registered leaf moves together.

In [ ]:
net = nnx.Sequential(nnx.Linear(X.shape[-1], 1, rngs=nnx.Rngs(0)))
nnx.update(net, jax.device_put(nnx.state(net), try_gpu()))
net(X)

The input arrived on the device, the parameters live on the device, so the
output is computed and stored there too. Let's confirm where the parameters
ended up.

In [ ]:
jax.tree_util.tree_map(lambda p: p.device, nnx.state(net))

Device hygiene is short in JAX because all state is explicit. The optimizer's
state is built from the model's parameter state, so it is
born wherever the parameters live; and inside a compiled function you never
place anything, since the computation runs where its inputs are. Keep the
parameters and each batch on one device and everything downstream follows.

## GPU Memory

Here is a puzzle that every JAX user hits in their first week. You create one
small array, yet `nvidia-smi` shows the card nearly full; is that a leak? It
is not, and the explanation is the right mental model for everything else in
this section. Requesting memory from the CUDA driver is slow, so on first use
XLA *preallocates* about 75% of the GPU's memory as one block and hands out
pieces of it from its own allocator (the fraction is controlled by the
`XLA_PYTHON_CLIENT_PREALLOCATE` and `XLA_PYTHON_CLIENT_MEM_FRACTION`
environment variables). `nvidia-smi` sees the process from the outside, so it
reports the whole preallocated block no matter how little of it your arrays
occupy. The real accounting lives inside: `device.memory_stats()` returns a
dictionary whose `bytes_in_use` entry counts live buffers and whose
`peak_bytes_in_use` entry is their high-water mark, the same nesting of views
that the figure sketches. On the CPU backend there is no such
allocator and the call returns `None`.

![The three memory views nest: nvidia-smi's driver allocation contains PyTorch's reserved cache, which contains the live tensors that memory_allocated() counts.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-allocator.svg)

In [ ]:
if num_gpus() > 0:
    def report(tag):
        s = gpu().memory_stats()
        print(f'{tag}: {s["bytes_in_use"] / 2**20:7.1f} MiB in use, '
              f'{s["peak_bytes_in_use"] / 2**20:7.1f} MiB peak')
    big = jnp.zeros((256, 1024, 1024))  # 1 GiB of float32
    big.block_until_ready()
    report('after creating big')
    del big
    report('after deleting it ')
else:
    print('No GPU: memory_stats() returns None on the CPU backend.')

On a GPU, `bytes_in_use` falls by a gibibyte the moment the buffer's last
reference dies, while `nvidia-smi` does not move at all: the block returns to
XLA's preallocated pool, not to the driver. If another process must share the
card, set `XLA_PYTHON_CLIENT_PREALLOCATE=false` in the environment before the
first computation; allocations then go to the driver on demand, visibly to
`nvidia-smi` but more slowly.

### What Fills Memory During Training

In that section we did the bookkeeping on paper: a model with
$N$ parameters costs $4N$ bytes for float32 weights, another $4N$ for their
gradients, and $8N$ for Adam's two moment estimates, roughly $16N$ before any
data arrives. The remaining term, the *activations*, is different in kind:
backpropagation must remember the intermediate results of the forward pass in
order to compute gradients, and their size scales with the batch size and with
the width of every layer, while the $16N$ does not.

We can watch the terms arrive by reading `bytes_in_use` around the pieces of
a training step. The activations are the exception: they exist only while the
gradient computation runs, so they leave their trace in `peak_bytes_in_use`
rather than in a reading you can take afterwards.

In [ ]:
if num_gpus() > 0:
    def in_use():
        s = gpu().memory_stats()
        return f'{s["bytes_in_use"] / 2**20:7.1f} MiB'
    rngs = nnx.Rngs(0)
    net = nnx.Sequential(nnx.Linear(1024, 4096, rngs=rngs), nnx.relu,
                         nnx.Linear(4096, 4096, rngs=rngs), nnx.relu,
                         nnx.Linear(4096, 4096, rngs=rngs), nnx.relu,
                         nnx.Linear(4096, 10, rngs=rngs))
    key1, key2 = jax.random.split(d2l.get_key())
    X = jax.random.normal(key1, (4096, 1024))
    y = jax.random.randint(key2, (4096,), 0, 10)
    def loss(model):
        logits = model(X)
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, y).mean()
    jax.block_until_ready(nnx.state(net))
    print('weights              ', in_use())
    grads = jax.block_until_ready(nnx.grad(loss)(net))
    print('+ gradients          ', in_use())
    optimizer = nnx.Optimizer(net, optax.adam(1e-3), wrt=nnx.Param)
    jax.block_until_ready(optimizer.opt_state)
    print('+ optimizer state    ', in_use())
    peak = gpu().memory_stats()['peak_bytes_in_use'] / 2**20
    print(f'peak with activations {peak:7.1f} MiB')
else:
    print('No GPU: run this cell on a GPU to see the terms arrive.')

The readings map onto the accounting. The first is the weights alone (about
$4N$ bytes, plus the input batch). The second adds gradients exactly the size
of the weights: in JAX a gradient is not a field attached to a parameter but
a second pytree of the same shape, returned by `jax.grad`. The third adds
$8N$ when `optax.adam`'s `init` builds its two moment pytrees from the
parameters, placed wherever they live. The activations never appear as a
plateau: they exist only inside the `jax.grad` call, so the peak reading,
several times the weights here because the batch is large, is the only
evidence they leave. The practical consequence is the same everywhere: when
you are out of memory, the knob that works immediately is the batch size,
because it scales the one term the model architecture does not fix. If a
*growing* staircase appears across steps, look for device arrays accumulating
on the Python side, such as appending the loss array itself, rather than
`loss.item()`, to a running log.

### Trading Compute for Memory: Activation Checkpointing

The activation term has a second knob. Backpropagation stores every
intermediate result only to read each one exactly once, on the way back.
*Activation checkpointing* [@Chen.Xu.Zhang.ea.2016] refuses to store
them: it keeps only the inputs to selected segments of the network, and during
the backward pass reruns each segment's forward computation to regenerate the
activations it needs. For a stack of $N$ equally sized layers, dividing the
stack into segments of about $\sqrt{N}$ layers reduces stored activations from
$O(N)$ to $O(\sqrt{N})$. The price is roughly one extra forward pass, often
30--40% more compute per step. the figure sketches
this schedule.
The trade pays off exactly where deep learning spends most of its time: deep
stacks of identical blocks, such as the residual stack we assembled in
that section and every Transformer you will meet later.
We rebuild a compact version of that block here.

![Standard backpropagation stores every activation (top); checkpointing keeps only segment-boundary activations and recomputes the rest during the backward pass (bottom), trading roughly 1.3x compute for O(sqrt(N)) instead of O(N) activation memory.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-activation-checkpoint.svg)

In [ ]:
class ResidualBlock(nnx.Module):  # As in :numref:`sec_model_construction`
    def __init__(self, num_hiddens, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu)

    def __call__(self, X):
        return X + self.body(X)

def run_stack(graphdef, states, X, use_checkpoint=False, segment_size=4):
    def segment(segment_states, X):
        for state in segment_states:
            X = nnx.merge(graphdef, state)(X)
        return X
    f = jax.checkpoint(segment) if use_checkpoint else segment
    for start in range(0, len(states), segment_size):
        X = f(states[start:start + segment_size], X)
    return X

`jax.checkpoint` (also exported as `jax.remat`) is a function transformation
like `grad` and `jit`: `jax.checkpoint(f)` behaves exactly like `f` when
called, but when differentiated it saves only `f`'s inputs and recomputes the
internals during the backward pass. Randomness needs no special handling:
JAX functions receive their PRNG keys as explicit arguments, so recomputation
with the same key reproduces a segment containing dropout bit for bit by
construction.

Before measuring memory, we should verify the claim that nothing about the
result changes: the gradients through a checkpointed stack must equal the
ordinary ones.

In [ ]:
X = jax.random.normal(jax.random.key(1), (32, 64))
blocks = [ResidualBlock(64, nnx.Rngs(k))
          for k in jax.random.split(jax.random.key(0), 4)]
split_blocks = [nnx.split(block) for block in blocks]
graphdef = split_blocks[0][0]
states = [state for _, state in split_blocks]

def stack_grads(use_checkpoint):
    loss = lambda states: run_stack(graphdef, states, X,
                                    use_checkpoint).sum()
    return jax.tree_util.tree_leaves(jax.grad(loss)(states))

for g, g_ckpt in zip(stack_grads(False), stack_grads(True)):
    assert jnp.allclose(g, g_ckpt)
print('checkpointed gradients match the ordinary ones')

That check runs anywhere, CPU included. The memory effect needs a GPU and a
stack deep enough for the difference to be unambiguous: sixteen blocks of
width 1024 at batch size 8192.

In [ ]:
if num_gpus() > 0:
    X = jax.random.normal(jax.random.key(1), (8192, 1024))
    blocks = [ResidualBlock(1024, nnx.Rngs(k))
              for k in jax.random.split(jax.random.key(0), 16)]
    split_blocks = [nnx.split(block) for block in blocks]
    graphdef = split_blocks[0][0]
    states = [state for _, state in split_blocks]
    for use_checkpoint in (True, False):  # low first: the peak only grows
        grad_fn = jax.jit(jax.grad(
            lambda states: run_stack(
                graphdef, states, X, use_checkpoint).sum()))
        jax.block_until_ready(grad_fn(states))  # compile outside timing
        t = time.time()
        jax.block_until_ready(grad_fn(states))
        peak = gpu().memory_stats()['peak_bytes_in_use'] / 2**20
        print(f'checkpointing={use_checkpoint!s:5}  peak {peak:6.0f} MiB, '
              f'{time.time() - t:.2f} sec')
else:
    print('No GPU: peak-memory comparison needs a GPU.')

Without checkpointing, the peak carries the activations of all sixteen blocks
at once; with four-block segments, it carries the four segment inputs plus the
recomputed activations of one segment, at the cost of a slower step. When a
model almost fits, this
trade is the difference between training and not training, which is why large
Transformer training runs use it as a matter of course.

## Don't Break the Pipeline

The device runs ahead of Python; in JAX this *asynchronous dispatch* is the
native execution model on every backend, the CPU included. When you write
`B = A @ A`, JAX does not wait for the multiplication: it queues the
computation, returns an array that is really a promise of the result, and
Python races on to enqueue the next operation. This asynchrony is where much
of the speed comes from, because Python can prepare work while the device
crunches. It also means that timing or logging naively measures nothing, or
worse, stalls everything. The explicit synchronization point is
`block_until_ready()`; with it we can see the gap between *queueing* work and
the work *finishing*, on this machine's CPU as well as on any GPU.

In [ ]:
A = jax.random.normal(jax.random.key(0), (2000, 2000)) / 45
(A @ A).block_until_ready()  # Warm up
B, t = A, time.time()
for _ in range(32):
    B = B @ A  # Chained, so the last result implies all finished
print(f'time to queue 32 matrix products: {time.time() - t:.4f} sec')
B.block_until_ready()
print(f'time until they all finished:     {time.time() - t:.4f} sec')

Python returned from the loop in a few milliseconds — far less time than
the thousand products take; they were still running. Any operation that needs a concrete value on the host forces a
*synchronization point*: `.item()`, `np.asarray`, `print`, an `if` on an
array's value, and `block_until_ready()` itself, whose job is to make the
synchronization explicit (our timings above depend on it). Each one makes
Python block until the queued work drains, and the device then sits idle
until Python catches up. A `print(loss.item())` in the inner loop can
serialize the whole pipeline this way, once per step, as
the figure lays out. The fix is not to give up monitoring
but to move it off the hot path: keep running statistics on the device and
transfer them once per epoch, or hand values to a background consumer, which
is exactly what our `ProgressBoard` from that section does when the
`Trainer` plots the loss.

![Python queues kernels k1 through k4 and races ahead while the GPU works through them serially; loss.item() forces a synchronization point where the CPU blocks until the GPU drains its queue, then both resume.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-async-queue.svg)

The same overlap idea applies to getting data *onto* the device, and in JAX
it needs no per-tensor flag: `device_put` dispatches the transfer and returns
immediately, so Python prepares the next batch while the copy overlaps the
device's current work, which is exactly the effect other stacks arrange with
pinned host buffers. The full treatment of asynchrony, streams, and
multi-device parallelism is in that section.

## The Trainer, Now with Devices

In that section the `Trainer` accepted a `num_gpus` argument and
ignored it. We can now redeem that promise with everything this section
taught: the parameters are created on the accelerator *once*, before training
(JAX initializes arrays on the default device, which is the accelerator
whenever one exists), and every batch streams over per step, at the boundary
of the computation.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus, d2l.num_gpus()))]

@d2l.add_to_class(d2l.Trainer)
def prepare_batch(self, batch):
    if self.gpus:
        batch = [d2l.to(a, self.gpus[0]) for a in batch]
    return batch

The `min(num_gpus, d2l.num_gpus())` makes the request a ceiling rather than a
demand: on a machine with no GPU, `self.gpus` is empty, `prepare_batch` is a
no-op, and training proceeds on the CPU. There is no model hook to patch,
because `fit` initializes the parameters itself and JAX creates them on the
default device, the GPU whenever one is present. Every `Trainer(num_gpus=1)`
you see from the next chapter onward relies on this fallback, which is how
one codebase serves both the laptop you are reading on and a GPU server. As a
capstone we train a classifier built from this chapter's residual blocks on
Fashion-MNIST; the `Trainer` moves each batch, and the memory budget of the
run is exactly the accounting from above.

In [ ]:
class ResMLPClassifier(d2l.Classifier):
    def __init__(self, num_hiddens=256, num_blocks=2, lr=0.1, rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.net = nnx.Sequential(
            lambda x: x.reshape((x.shape[0], -1)),
            nnx.Linear(784, num_hiddens, rngs=rngs), nnx.relu,
            *[ResidualBlock(num_hiddens, rngs) for _ in range(num_blocks)],
            nnx.Linear(num_hiddens, 10, rngs=rngs))

trainer = d2l.Trainer(max_epochs=1, num_gpus=1)
trainer.fit(ResMLPClassifier(), d2l.FashionMNIST(batch_size=256))

## Summary

Every array and every parameter lives on a device, and committed arrays
combine only when co-located; copies between devices are explicit
(`device_put`), slow relative to compute, and belong at the boundary of the
training loop, not inside it. GPU memory during training holds four things:
weights, gradients, optimizer state (fixed by the model and optimizer), and
activations (scaling with batch size); `device.memory_stats()` tracks live
buffers inside the block XLA preallocates, while `nvidia-smi` shows the whole
block, so the two disagree by design. Activation checkpointing
(`jax.checkpoint`) recomputes instead of stores, trading roughly a third more
compute for activation memory. Dispatch is asynchronous on every backend:
`.item()`, `np.asarray`, and `print` are synchronization points that stall
the pipeline, and `block_until_ready()` is the explicit one that accurate
timings need. The `Trainer` encodes the placement discipline: parameters
created on the device once, batches moved per step, graceful CPU fallback
when no GPU exists.

## Exercises

1. Using the accounting model of this section, predict the peak memory of the
   four-plateau cell at batch sizes 64, 256, and 1024, then measure with
   your framework's peak-memory counter. Where does the prediction break down,
   and what did it omit?
1. Increase the batch size in the checkpointing comparison until the
   non-checkpointed run raises an out-of-memory error. How much further can
   the checkpointed run go before it does too? Explain the ratio using the
   sizes of what each variant stores.
1. Time one epoch of the capstone training run with `print(loss.item())`
   after every step, and again printing only once per epoch. Explain the
   difference in terms of synchronization points.
1. If you have two GPUs, time 1000 matrix products of two $4096 \times 4096$
   matrices executed on one GPU, then 500 on each of two GPUs issued from the
   same loop. You should see almost linear scaling; explain why Python can
   drive both cards at once. Guard the experiment with `num_gpus() >= 2`.